In [26]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [27]:
# CELL 1: Install dependencies (no Flash Attention — compile issues on Colab)
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl
!pip install -q pillow-avif-plugin num2words

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [28]:
# CELL 2: Optional HF login (speeds up downloads, avoids rate limits)
# Uncomment the next two lines if you have a token
from huggingface_hub import login
login()

In [29]:
# CELL 3: Load plantvillage-full and inspect structure
from datasets import load_dataset
from collections import Counter

ds = load_dataset("geraldmc/plantvillage-full", revision="v0.1.0")
train_ds = ds["train"]

print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(f"Columns: {train_ds.column_names}")
print(f"Total samples: {len(train_ds)}")

ex = train_ds[0]
print("\nFirst sample fields:")
for k, v in ex.items():
    if k == "image":
        print(f"  {k}: PIL Image {v.size}")
    else:
        print(f"  {k}: {v}")

label_counts = Counter(train_ds["class_label"])
print(f"\nTotal classes: {len(label_counts)}")
print(f"Min samples/class: {min(label_counts.values())}")
print(f"Max samples/class: {max(label_counts.values())}")

DATASET STRUCTURE
Columns: ['image', 'class_label', 'class_idx', 'host', 'disease', 'split', 'leaf_id', 'leaf_grouped']
Total samples: 54304

First sample fields:
  image: PIL Image (256, 256)
  class_label: Apple___Apple_scab
  class_idx: 0
  host: Apple
  disease: Apple scab
  split: test
  leaf_id: Apple___Apple_scab::leaf_79
  leaf_grouped: True

Total classes: 38
Min samples/class: 152
Max samples/class: 5507


In [30]:
# CELL 4: Convert to ChatML format — keeps ONLY `image` and `messages` columns
def build_conversational_batch(batch):
    messages_list = []
    for label in batch["class_label"]:
        if "___" in label:
            host, disease = label.split("___", 1)
            disease_display = disease.replace("_", " ").strip()
        else:
            host, disease_display = "Unknown", label.replace("_", " ")

        if disease_display.lower() == "healthy":
            answer = f"**Plant:** {host}\n**Disease:** Healthy\n**Symptoms:** No visible disease symptoms."
        else:
            answer = f"**Plant:** {host}\n**Disease:** {disease_display}\n**Symptoms:** Visible symptoms consistent with {disease_display}."

        messages_list.append([
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": "Identify the plant disease in this image."}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        ])

    return {
        "messages": messages_list,
        "image": batch["image"],
    }

print("Converting to conversational format...")
converted = train_ds.map(
    build_conversational_batch,
    batched=True,
    batch_size=256,
    remove_columns=train_ds.column_names,
    desc="Formatting",
    load_from_cache_file=False,
)

print(f"Converted: {len(converted)}")
print(f"Columns: {converted.column_names}")

s = converted[0]
print(f"\nimage type: {type(s['image'])}")
print(f"image size: {s['image'].size}")
print(f"messages turns: {len(s['messages'])}")
print(f"\nFirst sample messages:\n{s['messages']}")

Converting to conversational format...


Formatting:   0%|          | 0/54304 [00:00<?, ? examples/s]

Converted: 54304
Columns: ['image', 'messages']

image type: <class 'PIL.JpegImagePlugin.JpegImageFile'>
image size: (256, 256)
messages turns: 2

First sample messages:
[{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'Identify the plant disease in this image.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '**Plant:** Apple\n**Disease:** Apple scab\n**Symptoms:** Visible symptoms consistent with Apple scab.'}]}]


In [31]:
# CELL 5: Balanced subset — 60 per class keeps training under ~10 min
import random
from collections import defaultdict

random.seed(42)

class_indices = defaultdict(list)
for i, sample in enumerate(converted):
    answer = sample["messages"][1]["content"][0]["text"]
    lines = answer.split("\n")
    plant = lines[0].replace("**Plant:** ", "").strip()
    disease = lines[1].replace("**Disease:** ", "").strip()
    class_indices[f"{plant}___{disease}"].append(i)

MAX_PER_CLASS = 60      # ← reduced from 120 for speed
selected = []
for cls, idxs in class_indices.items():
    random.shuffle(idxs)
    selected.extend(idxs[:MAX_PER_CLASS])

random.shuffle(selected)
subset = converted.select(selected)

split = subset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data = split["test"]

print(f"Subset: {len(subset)} | Train: {len(train_data)} | Val: {len(val_data)}")
print(f"Classes: {len(class_indices)}")
print(f"Expected steps/epoch (batch=1, accum=8): {len(train_data) // 8}")

Subset: 2280 | Train: 2052 | Val: 228
Classes: 38
Expected steps/epoch (batch=1, accum=8): 256


In [32]:
# CELL 6: Load SmolVLM-500M with 4-bit quantization + LoRA
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Processor with reduced image resolution (biggest speed win)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    size={"longest_edge": 512},   # default is 2048 → ~16× fewer vision tokens
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading model with 4-bit quantization...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
    init_lora_weights="gaussian",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model loaded successfully.")

Loading model with 4-bit quantization...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

trainable params: 9,568,256 || all params: 517,050,560 || trainable%: 1.8505
Model loaded successfully.


In [33]:
# CELL 7: Training — 1 epoch, no gradient checkpointing, no tf32 (T4 is Turing, not Ampere)
from trl import SFTConfig, SFTTrainer
from PIL import Image

def collate_fn(examples):
    texts = []
    images = []
    for ex in examples:
        text = processor.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        img = ex.get("image", ex.get("images"))
        if isinstance(img, list):
            img = img[0]
        if not isinstance(img, Image.Image):
            raise TypeError(f"Expected PIL Image, got {type(img)}")
        images.append(img)

    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=768,
    )
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    return batch

per_device_batch = 1
grad_accum = 8
num_epochs = 1
effective_batch = per_device_batch * grad_accum

steps_per_epoch = len(train_data) // effective_batch
total_steps = steps_per_epoch * num_epochs
warmup_steps = max(1, int(total_steps * 0.03))

print(f"Steps/epoch: {steps_per_epoch} | Total: {total_steps} | Warmup: {warmup_steps}")

training_args = SFTConfig(
    output_dir="./smolvlm-plantvillage-output",
    num_train_epochs=num_epochs,
    per_device_train_batch_size=per_device_batch,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-4,
    warmup_steps=warmup_steps,
    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",
    gradient_checkpointing=False,     # 500M fits on T4 without it
    bf16=True,
    # tf32=True,                      # ← REMOVED: T4 is Turing, not Ampere
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    max_length=768,
    loss_type="nll",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=None,
    data_collator=collate_fn,
    processing_class=processor,
)

print("Starting training...")
trainer.train()

adapter_dir = "./smolvlm-plantvillage-lora"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 49279, 'bos_token_id': 1}.


Steps/epoch: 256 | Total: 256 | Warmup: 7
Starting training...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,9.171246
20,0.938710
30,0.211382
40,0.131470
50,0.081897
60,0.061998
70,0.053393
80,0.044576
90,0.037195
100,0.029503


Adapter saved to ./smolvlm-plantvillage-lora


In [34]:
# CELL 8: Quick inference check on a validation sample
import torch

test_sample = val_data[0]
test_image = test_sample["image"]

messages = [{
    "role": "user",
    "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]
}]

model.eval()
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[test_image], return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)

response = processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("MODEL OUTPUT")
print("=" * 60)
print(response)
print("\n" + "=" * 60)
print("GROUND TRUTH")
print("=" * 60)
print(test_sample["messages"][1]["content"][0]["text"])

MODEL OUTPUT
 **Plant:** Potato
**Disease:** Late blight
**Symptoms:** Visible symptoms consistent with Late blight.

GROUND TRUTH
**Plant:** Potato
**Disease:** Late blight
**Symptoms:** Visible symptoms consistent with Late blight.


In [42]:
# EVALUATION: Check real accuracy on validation set
from PIL import Image
import torch
from tqdm import tqdm

model.eval()
correct = 0
total = 0

for i in tqdm(range(min(100, len(val_data))), desc="Evaluating"):
    sample = val_data[i]
    img = sample["image"]
    gt = sample["messages"][1]["content"][0]["text"]

    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Identify the plant disease in this image."}
    ]}]

    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    pred = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Extract disease line from both
    def get_disease(text):
        for line in text.split("\n"):
            if "Disease:" in line:
                return line.replace("**Disease:**", "").strip().lower()
        return "unknown"

    if get_disease(pred) == get_disease(gt):
        correct += 1
    total += 1

print(f"\nValidation accuracy (disease name): {correct}/{total} = {correct/total*100:.1f}%")

Evaluating: 100%|██████████| 100/100 [07:00<00:00,  4.21s/it]


Validation accuracy (disease name): 72/100 = 72.0%


In [37]:
# CELL 9: Merge LoRA adapter into base model (with full torchao bypass + fallback + sanity check)
import torch, os, json

# ============================================================
# STEP 1: Comprehensive torchao bypass
# ============================================================
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False

try:
    import peft.tuners.lora.torchao as _lora_torchao
    _lora_torchao.is_torchao_available = lambda: False
except Exception:
    pass

try:
    import peft.tuners.lora.model as _lora_model
    if hasattr(_lora_model, "is_torchao_available"):
        _lora_model.is_torchao_available = lambda: False
except Exception:
    pass

print("[1/4] torchao bypass applied.")

# ============================================================
# STEP 2: Load base model in fp16 on CPU
# ============================================================
from transformers import AutoModelForImageTextToText, AutoProcessor

print("[2/4] Loading base model in fp16 for merging...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)

merged_dir = "./smolvlm-plantvillage-merged"
merge_success = False

# ============================================================
# STEP 3: Try PEFT merge first, fall back to manual merge
# ============================================================
print("[3/4] Attempting PEFT merge...")
try:
    from peft import PeftModel
    merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
    merged_model = merged_model.merge_and_unload()
    merged_model.save_pretrained(merged_dir)
    processor.save_pretrained(merged_dir)
    print(f"  ✓ PEFT merge succeeded → {merged_dir}")
    merge_success = True
except Exception as e:
    print(f"  ✗ PEFT merge failed: {type(e).__name__}: {e}")
    print("  → Falling back to manual safetensors merge...")

# ============================================================
# STEP 3b: Manual merge fallback
# ============================================================
if not merge_success:
    from safetensors.torch import load_file

    adapter_file = os.path.join(adapter_dir, "adapter_model.safetensors")
    if not os.path.exists(adapter_file):
        raise FileNotFoundError(f"Adapter not found at {adapter_file}")

    lora_state = load_file(adapter_file)
    with open(os.path.join(adapter_dir, "adapter_config.json")) as f:
        adapter_cfg = json.load(f)

    r = adapter_cfg["r"]
    alpha = adapter_cfg["lora_alpha"]
    scale = alpha / r
    print(f"  LoRA config: rank={r}, alpha={alpha}, scale={scale:.3f}")

    sd = base_model.state_dict()
    merged_count = 0
    skipped = 0

    for key in list(lora_state.keys()):
        if ".lora_A." not in key:
            continue
        # ...layers.0.self_attn.q_proj.lora_A.weight
        #  -> ...layers.0.self_attn.q_proj.weight
        base_key = key.replace(".lora_A.weight", ".weight")
        b_key = key.replace(".lora_A.weight", ".lora_B.weight")

        if b_key not in lora_state or base_key not in sd:
            skipped += 1
            continue

        lora_A = lora_state[key].to(torch.float16)
        lora_B = lora_state[b_key].to(torch.float16)
        delta = (lora_B @ lora_A) * scale

        sd[base_key] = sd[base_key] + delta.to(sd[base_key].dtype)
        merged_count += 1

    print(f"  Merged {merged_count} LoRA pairs, skipped {skipped}.")

    base_model.load_state_dict(sd, strict=False)
    base_model.save_pretrained(merged_dir)

    processor = AutoProcessor.from_pretrained(adapter_dir)
    processor.save_pretrained(merged_dir)
    print(f"  ✓ Manual merge saved → {merged_dir}")

# ============================================================
# STEP 4: Verify merged model
# ============================================================
print("[4/4] Verifying merged model...")
files = os.listdir(merged_dir)
print(f"  Files: {sorted(files)}")

st_path = os.path.join(merged_dir, "model.safetensors")
if os.path.exists(st_path):
    size_gb = os.path.getsize(st_path) / 1e9
    print(f"  model.safetensors size: {size_gb:.2f} GB")
    if size_gb < 0.7:
        print("  ⚠ WARNING: file seems too small — merge may have failed")
    else:
        print("  ✓ Size looks correct")
else:
    print("  ⚠ model.safetensors not found — check other shard names")

print("=" * 60)
print("CELL 9 COMPLETE")
print("=" * 60)
print(f"Merged model ready at: {merged_dir}")
print("Next: run Cell 10 (build llama.cpp)")

[1/4] torchao bypass applied.
[2/4] Loading base model in fp16 for merging...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

[3/4] Attempting PEFT merge...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ PEFT merge succeeded → ./smolvlm-plantvillage-merged
[4/4] Verifying merged model...
  Files: ['chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']
  model.safetensors size: 1.02 GB
  ✓ Size looks correct
CELL 9 COMPLETE
Merged model ready at: ./smolvlm-plantvillage-merged
Next: run Cell 10 (build llama.cpp)


In [43]:
# CELL 10: Verify or build llama.cpp in /kaggle/working
import os

os.chdir("/kaggle/working")
llama_dir = "/kaggle/working/llama.cpp"
quantize_bin = "/kaggle/working/llama.cpp/build/bin/llama-quantize"
convert_script = "/kaggle/working/llama.cpp/convert_hf_to_gguf.py"

if os.path.exists(llama_dir):
    print(f"✓ llama.cpp exists at {llama_dir}")
    print(f"  convert script: {os.path.exists(convert_script)}")
    print(f"  quantize binary: {os.path.exists(quantize_bin)}")
else:
    print("✗ llama.cpp not found — cloning and building now...")
    !git clone https://github.com/ggerganov/llama.cpp
    !cd llama.cpp && pip install -q -r requirements.txt
    !cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF
    !cd llama.cpp && cmake --build build --config Release -j4
    print("✓ Build complete")

# Sanity check
for p in [convert_script, quantize_bin]:
    if os.path.exists(p):
        print(f"  ✅ {p}")
    else:
        print(f"  ❌ MISSING: {p}")

✓ llama.cpp exists at /kaggle/working/llama.cpp
  convert script: True
  quantize binary: True
  ✅ /kaggle/working/llama.cpp/convert_hf_to_gguf.py
  ✅ /kaggle/working/llama.cpp/build/bin/llama-quantize


In [38]:
# CELL 10: Clone llama.cpp and build tools
import os

if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp

!cd llama.cpp && pip install -q -r requirements.txt
!cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF
!cd llama.cpp && cmake --build build --config Release -j4
print("llama.cpp built.")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 137420, done.
remote: Counting objects: 100% (412/412), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 137420 (delta 298), reused 197 (delta 197), pack-reused 137008 (from 2)
Receiving objects: 100% (137420/137420), 445.59 MiB | 18.44 MiB/s, done.
Resolving deltas: 100% (97697/97697), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 96.7 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [39]:
# CELL 11: Convert HF model to GGUF (F16 first)
!cd llama.cpp && python convert_hf_to_gguf.py ../smolvlm-plantvillage-merged \
    --outfile ../smolvlm-plantvillage-f16.gguf --outtype f16

!cd llama.cpp && python convert_hf_to_gguf.py ../smolvlm-plantvillage-merged \
    --mmproj --outfile ../mmproj-smolvlm-f16.gguf --outtype f16

!ls -lh smolvlm-plantvillage-f16.gguf mmproj-smolvlm-f16.gguf

INFO:hf-to-gguf:Loading model: smolvlm-plantvillage-merged
INFO:hf-to-gguf:Model architecture: VLlama3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {960, 49280}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {960, 49280}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {960}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {2560, 960}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {960, 2560}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {960, 2560}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {960}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {960, 320}
INFO:hf-to-gguf:blk.0.attn_

In [40]:
# CELL 12: Quantize for Android deployment
!cd llama.cpp && ./build/bin/llama-quantize \
    ../smolvlm-plantvillage-f16.gguf \
    ../smolvlm-plantvillage-Q4_K_M.gguf Q4_K_M

!cd llama.cpp && ./build/bin/llama-quantize \
    ../mmproj-smolvlm-f16.gguf \
    ../mmproj-smolvlm-Q8_0.gguf Q8_0

!ls -lh smolvlm-plantvillage-Q4_K_M.gguf mmproj-smolvlm-Q8_0.gguf

version: 0.6.0-dev (build 11503, commit 03aa006ac)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '../smolvlm-plantvillage-f16.gguf' to '../smolvlm-plantvillage-Q4_K_M.gguf' as Q4_K_M
gguf_init_from_file: failed to open GGUF file '../smolvlm-plantvillage-f16.gguf' (No such file or directory)
llama_model_quantize: failed to quantize: llama_model_loader: failed to load model from ../smolvlm-plantvillage-f16.gguf
llama_quantize: failed to quantize model from '../smolvlm-plantvillage-f16.gguf'
version: 0.6.0-dev (build 11503, commit 03aa006ac)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '../mmproj-smolvlm-f16.gguf' to '../mmproj-smolvlm-Q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 21 key-value pairs and 198 tensors from ../mmproj-smolvlm-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.archi

In [41]:
# CELL 13: Package everything for Android + download
!mkdir -p /content/android_export
!cp smolvlm-plantvillage-Q4_K_M.gguf /content/android_export/
!cp mmproj-smolvlm-Q8_0.gguf /content/android_export/
!cp -r smolvlm-plantvillage-lora /content/android_export/ 2>/dev/null || true
!cd /content && zip -r smolvlm-plantvillage-android.zip android_export/

print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print("Files ready for Android:")
print("  smolvlm-plantvillage-Q4_K_M.gguf   (main model)")
print("  mmproj-smolvlm-Q8_0.gguf           (vision projector)")
print("\nBoth files MUST be loaded together in your Android app")
print("via the mmprojPath parameter of llamafu / mt_llmkit.")

# Auto-download the zip
from google.colab import files
files.download("/content/smolvlm-plantvillage-android.zip")

cp: cannot stat 'smolvlm-plantvillage-Q4_K_M.gguf': No such file or directory
  adding: android_export/ (stored 0%)
  adding: android_export/mmproj-smolvlm-Q8_0.gguf (deflated 7%)
  adding: android_export/smolvlm-plantvillage-lora/ (stored 0%)
  adding: android_export/smolvlm-plantvillage-lora/tokenizer.json (deflated 82%)
  adding: android_export/smolvlm-plantvillage-lora/adapter_config.json (deflated 60%)
  adding: android_export/smolvlm-plantvillage-lora/tokenizer_config.json (deflated 50%)
  adding: android_export/smolvlm-plantvillage-lora/chat_template.jinja (deflated 51%)
  adding: android_export/smolvlm-plantvillage-lora/README.md (deflated 65%)
  adding: android_export/smolvlm-plantvillage-lora/processor_config.json (deflated 59%)
  adding: android_export/smolvlm-plantvillage-lora/adapter_model.safetensors (deflated 20%)
EXPORT COMPLETE
Files ready for Android:
  smolvlm-plantvillage-Q4_K_M.gguf   (main model)
  mmproj-smolvlm-Q8_0.gguf           (vision projector)

Both files 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>